In [2]:
import pandas as pd
import numpy as np

file_path = "/Users/amirmohammadalinaghi/Documents/JOB 2026/Simple Germany/ING Risk /Project/Data/train.csv"

df = pd.read_csv(file_path, low_memory=False)

df.shape

(215257, 102)

In [3]:
df.head()

,ID,NAME_CONTRACT_TYPE,GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,EXT_SOURCE_4,TARGET
0,160132,Cash loans,F,N,Y,0,427500.0,1288350.0,37800.0,1125000.0,...,0,0,0.0,0.0,1.0,0.0,0.0,5.0,0.242982,0
1,233132,Cash loans,M,Y,Y,0,180000.0,848745.0,40963.5,675000.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0.115508,0
2,307771,Cash loans,M,Y,Y,1,112500.0,385164.0,19795.5,292500.0,...,0,0,0.0,0.0,0.0,0.0,0.0,4.0,0.393106,0
3,376452,Cash loans,F,N,Y,0,540000.0,1433520.0,60867.0,1237500.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.455985,0
4,405403,Cash loans,M,Y,Y,0,76500.0,900000.0,26316.0,900000.0,...,0,0,0.0,0.0,0.0,3.0,0.0,2.0,0.141508,0


In [4]:
df.columns.tolist()

['ID',
 'NAME_CONTRACT_TYPE',
 'GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'DAYS_AGE',
 'DAYS_EMPLOYMENT',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'YEARS_BUILD_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'NON

In [28]:
risk_df = df[
    [
        "TARGET",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "DAYS_AGE",
        "DAYS_EMPLOYMENT",
        "CNT_CHILDREN",
        "NAME_INCOME_TYPE",
        "NAME_EDUCATION_TYPE",
        "NAME_FAMILY_STATUS",
        "OCCUPATION_TYPE",
        "REGION_RATING_CLIENT",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]
].copy()


In [17]:
#Creating Risk Metrics:
#Age
risk_df["AGE"] = abs(risk_df["DAYS_AGE"]) / 365
#Employment_Years:
risk_df["EMPLOYMENT_YEARS"] = abs(risk_df["DAYS_EMPLOYMENT"]) / 365
#Debt_to_Income Ratio
risk_df["DTI"] = (
    risk_df["AMT_ANNUITY"] * 12
) / risk_df["AMT_INCOME_TOTAL"]

In [18]:
#Portfolio_Default_Rate
default_rate = risk_df["TARGET"].mean()

print(f"Default Rate: {default_rate:.2%}")

Default Rate: 8.11%


In [19]:
#Insight 
#The overall portfolio default rate is 8.1%, indicating that approximately one in twelve applicants experienced repayment difficulties. 
#This serves as the baseline credit risk level against which all borrower segments and stress scenarios will be evaluated.

In [20]:
#Income Segmentation
risk_df["income_group"] = pd.qcut(
    risk_df["AMT_INCOME_TOTAL"],
    q=5,
    labels=["Very Low","Low","Medium","High","Very High"]
)

risk_df.groupby("income_group")["TARGET"].mean()

income_group
Very Low     0.082236
Low          0.086528
Medium       0.085842
High         0.081544
Very High    0.065161
Name: TARGET, dtype: float64

In [ ]:
#Risk Insight
#Default rates remain relatively stable across most income segments, ranging between 8.1% and 8.7%.
#Only borrowers in the highest income quintile exhibit a noticeably lower default rate of 6.5%.
#This suggests that while higher income provides some protection against default, income alone is not a strong discriminator of credit risk
#within the portfolio.

In [21]:
#DTI Segmentation
risk_df["dti_group"] = pd.qcut(
    risk_df["DTI"],
    q=5,
    duplicates="drop"
)

risk_df.groupby("dti_group")["TARGET"].mean()

dti_group
(0.0643, 1.245]    0.072520
(1.245, 1.727]     0.077647
(1.727, 2.229]     0.081353
(2.229, 2.965]     0.088202
(2.965, 22.512]    0.085577
Name: TARGET, dtype: float64

In [ ]:
#Risk Insight
#Default rates generally increase as debt burden rises, with borrowers in the fourth DTI quintile exhibiting the highest 
#default rate of 8.8%.
#This pattern supports the hypothesis that higher repayment obligations reduce financial flexibility and increase vulnerability to
#payment difficulties. However, the highest DTI segment does not demonstrate the highest default rate,
#suggesting the relationship is not strictly linear.

In [26]:
#External Credit Score Analysis
risk_df.groupby(
    pd.qcut(risk_df["EXT_SOURCE_2"], 5)
)["TARGET"].mean()

EXT_SOURCE_2
(-0.0009999183, 0.34]    0.152983
(0.34, 0.512]            0.091343
(0.512, 0.609]           0.070413
(0.609, 0.682]           0.054125
(0.682, 0.855]           0.036361
Name: TARGET, dtype: float64

In [32]:
#Logistic Regression PD Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix

features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "REGION_RATING_CLIENT"
]

model_df = risk_df[features + ["TARGET"]].copy()

model_df = model_df.replace([np.inf, -np.inf], np.nan)

X = model_df[features]
y = model_df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

logit_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logit_model.fit(X_train, y_train)

y_pred_proba = logit_model.predict_proba(X_test)[:, 1]
y_pred = logit_model.predict(X_test)

auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)

print(f"AUC: {auc:.3f}")
print(f"Accuracy: {accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

AUC: 0.694
Accuracy: 0.656

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.66      0.78     59344
           1       0.14      0.61      0.22      5234

    accuracy                           0.66     64578
   macro avg       0.54      0.64      0.50     64578
weighted avg       0.89      0.66      0.73     64578


Confusion Matrix:
[[39162 20182]
 [ 2017  3217]]


In [ ]:
#Risk Insight
#A logistic regression model was developed to estimate borrower Probability of Default (PD) using financial, 
#demographic, employment, and external credit score variables. The model achieved an AUC of 0.694, indicating 
#moderate discriminatory power between defaulting and non-defaulting borrowers. 
#The results suggest that the selected borrower characteristics contain meaningful predictive information and can
#support risk-based credit assessment and portfolio monitoring activities.


#Interpretation
#The model successfully identified approximately 61% of defaulting borrowers, demonstrating a reasonable ability
#to detect elevated credit risk. Given the highly imbalanced nature of the portfolio, where only 8.1% of borrowers
#experienced repayment difficulties, model evaluation focused primarily on discriminatory performance rather than overall accuracy.
#While the model generated a relatively high number of false positive classifications, this behavior is often acceptable 
#within a risk management context, where failing to identify risky borrowers may be more costly than conducting additional 
#reviews of low-risk applicants.